# 01 - Banco de dados -> Landing

Extrai as tabelas do PostgreSQL (Supabase) e grava como CSV no volume `workspace.landing.dados`.

## Credenciais

As credenciais **não ficam mais no código**. O notebook procura, nesta ordem:

1. Secret Scope `lakehouse` (chaves `db_host`, `db_port`, `db_name`, `db_user`, `db_password`)
2. Widgets / parâmetros do Job com esses mesmos nomes

Criando o scope pela Databricks CLI:

```
databricks secrets create-scope lakehouse
databricks secrets put-secret lakehouse db_password
```

In [ ]:
SECRET_SCOPE = "lakehouse"
CONFIG_KEYS = ["db_host", "db_port", "db_name", "db_user", "db_password"]

for key in CONFIG_KEYS:
    dbutils.widgets.text(key, "")


def get_config(key, default=None):
    """Secret Scope primeiro; se não existir, usa o widget/parâmetro do Job."""
    try:
        return dbutils.secrets.get(SECRET_SCOPE, key)
    except Exception:
        pass
    value = dbutils.widgets.get(key)
    if value:
        return value
    if default is not None:
        return default
    raise ValueError(f"Configuração '{key}' não encontrada (Secret Scope '{SECRET_SCOPE}' ou widget).")


HOST     = get_config("db_host")
PORT     = get_config("db_port", "5432")
DATABASE = get_config("db_name", "postgres")
USER     = get_config("db_user")
PASS     = get_config("db_password")

url = f"jdbc:postgresql://{HOST}:{PORT}/{DATABASE}?sslmode=require"

props = {
    "user": USER,
    "password": PASS,
    "driver": "org.postgresql.Driver"
}

## Testando a conexão

In [ ]:
spark.read.jdbc(url, "games", properties=props).show(5)

## Lendo cada tabela, gravando CSV na landing-zone e renomeando

O Spark grava uma pasta com um arquivo `part-*.csv`. Para cada tabela: grava em `temp_<tabela>`, copia o `part-*` para `<tabela>.csv` e apaga a pasta temporária.

In [ ]:
landing_path = "/Volumes/workspace/landing/dados"

SOURCE_TABLES = ["games", "steam_metrics", "regional_sales", "how_long_to_beat", "walkthrough_links"]

for table in SOURCE_TABLES:
    tmp_dir = f"{landing_path}/temp_{table}"

    (spark.read.jdbc(url, table, properties=props)
          .coalesce(1)
          .write.mode("overwrite")
          .options(header=True, delimiter=",")
          .csv(tmp_dir))

    part_file = [f.path for f in dbutils.fs.ls(tmp_dir) if f.name.startswith("part-")][0]
    dbutils.fs.cp(part_file, f"{landing_path}/{table}.csv")
    dbutils.fs.rm(tmp_dir, recurse=True)
    print(f"OK  {table}.csv")

display(dbutils.fs.ls(landing_path))